# Детекция ботов

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

from metric import precision_at_recall

RANDOM_STATE = 42
DATA = 'data/'

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

## Загрузка

In [ ]:
DATE_COLS = ['cookie_created_at', 'window_start_ts', 'window_end_ts']

train = pd.read_csv(DATA + 'train.csv', parse_dates=DATE_COLS)
test = pd.read_csv(DATA + 'test.csv', parse_dates=DATE_COLS)
events_raw = pd.read_csv(DATA + 'events.csv', parse_dates=['event_ts'])

print(train.shape, test.shape, events_raw.shape)
events_raw.dtypes

In [ ]:
# сравнение дат строками отработает лексикографически и не упадёт, поэтому проверяем явно
assert all(pd.api.types.is_datetime64_any_dtype(train[c]) for c in DATE_COLS)
assert all(pd.api.types.is_datetime64_any_dtype(test[c]) for c in DATE_COLS)
assert pd.api.types.is_datetime64_any_dtype(events_raw.event_ts)

## Метаданные кук

In [ ]:
print('доля ботов:', round(train.target.mean(), 4))
print(train.target.value_counts().to_string())

In [ ]:
print('дубли cookie_id в train:', train.cookie_id.duplicated().sum())
print('дубли cookie_id в test :', test.cookie_id.duplicated().sum())
print('пересечение train/test :', len(set(train.cookie_id) & set(test.cookie_id)))

print()
print(pd.concat([train.isna().mean().rename('train'),
                 test.isna().mean().rename('test')], axis=1).round(4).to_string())

In [ ]:
for name, df in [('train', train), ('test', test)]:
    length = (df.window_end_ts - df.window_start_ts).unique()
    print(f'{name}: {len(df)} окон, длина {length}, '
          f'{df.window_start_ts.min().date()} .. {df.window_start_ts.max().date()}')
    print(f'{name}: кук создано позже начала окна:', (df.cookie_created_at > df.window_start_ts).sum())

Train покрывает 06.04-19.04, test 20.04-26.04. Пересечений нет ни по кукам, ни по времени,
значит валидация только временная.

## Пропуски в событиях

In [ ]:
events_raw.isna().mean().round(4)

In [ ]:
VALUE_COLS = ['item_id', 'item_category', 'item_location', 'seller_type',
              'search_query', 'search_page', 'pointer_x', 'pointer_y']

# пропуски структурные: поле заполнено только для своих типов событий
events_raw.groupby('event_name')[VALUE_COLS].apply(lambda d: d.isna().mean()).round(3)

In [ ]:
# pointer_x/y зависят не от типа события, а от платформы
platform_preview = events_raw.platform.str.lower().replace({'desktop': 'web', 'iphone': 'ios'})
pd.crosstab(platform_preview, events_raw.pointer_x.isna(), normalize='index').round(3)

Импутация тут неуместна: пустой `search_query` у `item_view` означает «неприменимо».
Учитывать на уровне агрегации.

## Качество файла событий

In [ ]:
print('eid -> event_name однозначно:', (events_raw.groupby('eid').event_name.nunique() == 1).all())
events_raw.groupby(['eid', 'event_name']).size()

In [ ]:
print('полных дублей строк:', events_raw.duplicated().sum())
print('дублей по (cookie, ts, eid):', events_raw.duplicated(subset=['cookie_id', 'event_ts', 'eid']).sum())

In [ ]:
print('отсортирован по event_ts:', events_raw.event_ts.is_monotonic_increasing)

sorted_within = events_raw.groupby('cookie_id').event_ts.apply(lambda s: s.is_monotonic_increasing)
print(f'кук с упорядоченными событиями: {sorted_within.sum()} из {len(sorted_within)}')

In [ ]:
ev_cookies = set(events_raw.cookie_id)
print('кук в events:', len(ev_cookies))
print('train без событий:', (~train.cookie_id.isin(ev_cookies)).sum())
print('test без событий :', (~test.cookie_id.isin(ev_cookies)).sum())
print('диапазон event_ts:', events_raw.event_ts.min(), '..', events_raw.event_ts.max())

In [ ]:
print(events_raw.platform.value_counts().to_string())
print()
for c in ['item_category', 'item_location', 'seller_type', 'user_agent']:
    print(c, events_raw[c].nunique())

`platform` записана в 11 вариантах трёх значений, частоты внутри группы почти равны.

## Очистка

In [ ]:
PLATFORM_MAP = {
    'web': 'web', 'desktop': 'web',
    'android': 'android',
    'ios': 'ios', 'iphone': 'ios',
}


def clean_events(df):
    out = df.copy()
    out['platform'] = out.platform.str.lower().map(PLATFORM_MAP)
    assert out.platform.notna().all()

    out = out.drop_duplicates()
    # порядок в исходном файле произвольный, а признакам на разницах времени нужна сортировка
    return out.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)


events = clean_events(events_raw)
print(len(events_raw), '->', len(events))
events.platform.value_counts()

## Окно наблюдения

`window_start_ts <= event_ts < window_end_ts`, правая граница строгая.

In [ ]:
def attach_window(events, meta):
    assert not meta.cookie_id.duplicated().any()
    return events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']],
                        on='cookie_id', how='inner')


def events_in_window(events, meta):
    ev = attach_window(events, meta)
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)].reset_index(drop=True)

In [ ]:
for name, meta in [('train', train), ('test', test)]:
    ev = attach_window(events, meta)
    before = (ev.event_ts < ev.window_start_ts).sum()
    after = (ev.event_ts >= ev.window_end_ts).sum()
    print(f'{name}: всего {len(ev)}, до окна {before}, после окна {after}, '
          f'отрезаем {round((before + after) / len(ev) * 100, 1)}%')

In [ ]:
ev_train = events_in_window(events, train)
ev_test = events_in_window(events, test)

for name, meta, ev in [('train', train, ev_train), ('test', test, ev_test)]:
    print(f'{name}: кук без событий в окне', (~meta.cookie_id.isin(set(ev.cookie_id))).sum(), 'из', len(meta))

print(len(ev_train), len(ev_test))

Из train отрезается ~17% событий, и все они после конца окна. Из test не отрезается ничего.
Без фильтра признаки train считались бы по более длинной истории, чем признаки test.

## train против test

In [ ]:
def share_table(col):
    a = ev_train[col].value_counts(normalize=True, dropna=False)
    b = ev_test[col].value_counts(normalize=True, dropna=False)
    t = pd.concat([a.rename('train'), b.rename('test')], axis=1).fillna(0)
    t['diff'] = t.test - t.train
    return t.sort_values('train', ascending=False).round(4)


for col in ['event_name', 'platform', 'seller_type', 'item_category']:
    print(share_table(col).to_string(), '\n')

In [ ]:
cnt_train = ev_train.groupby('cookie_id').size().reindex(train.cookie_id, fill_value=0)
cnt_test = ev_test.groupby('cookie_id').size().reindex(test.cookie_id, fill_value=0)

pd.concat([cnt_train.describe().rename('train'),
           cnt_test.describe().rename('test')], axis=1).round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

bins = np.logspace(0, np.log10(max(cnt_train.max(), cnt_test.max()) + 1), 40)
axes[0].hist(cnt_train, bins=bins, alpha=0.6, density=True, label='train')
axes[0].hist(cnt_test, bins=bins, alpha=0.6, density=True, label='test')
axes[0].set_xscale('log')
axes[0].set_title('событий на куку')
axes[0].legend()

s = share_table('event_name')
axes[1].barh(s.index, s.train, alpha=0.6, label='train')
axes[1].barh(s.index, s.test, alpha=0.6, label='test')
axes[1].set_title('типы событий')
axes[1].invert_yaxis()
axes[1].legend()

axes[2].hist(ev_train.event_ts.dt.hour, bins=24, range=(0, 24), alpha=0.6, density=True, label='train')
axes[2].hist(ev_test.event_ts.dt.hour, bins=24, range=(0, 24), alpha=0.6, density=True, label='test')
axes[2].set_title('час суток')
axes[2].legend()

plt.tight_layout()
plt.show()

## Признаки

Четыре группы: объём и состав активности, ритм событий, разнообразие просмотренного,
поведение курсора.

In [ ]:
CAT_COLS = ['event_name', 'platform', 'seller_type', 'item_category']


def category_shares(ev, col):
    counts = pd.crosstab(ev.cookie_id, ev[col].fillna('NA'))
    shares = counts.div(counts.sum(axis=1), axis=0)
    shares.columns = [f'{col}={v}' for v in shares.columns]
    return shares


def entropy(counts):
    p = counts / counts.sum()
    p = p[p > 0]
    return -(p * np.log(p)).sum()

In [ ]:
def f_base(ev):
    """Объём активности и её состав по категориальным полям."""
    volume = np.log1p(ev.groupby('cookie_id').size()).rename('log_n_events')
    return pd.concat([volume] + [category_shares(ev, c) for c in CAT_COLS], axis=1)


def f_time(ev):
    """Интервалы между событиями и распределение активности по часам."""
    g = ev.groupby('cookie_id')
    d = ev.assign(dt=g.event_ts.diff().dt.total_seconds()).groupby('cookie_id').dt

    out = pd.DataFrame({
        'dt_mean': d.mean(),
        'dt_median': d.median(),
        'dt_min': d.min(),
        'dt_p10': d.quantile(0.10),
        'dt_p90': d.quantile(0.90),
        'frac_dt_lt2': d.apply(lambda s: (s < 2).mean()),
    })
    out['dt_cv'] = d.std() / out.dt_mean.replace(0, np.nan)
    out['rate'] = g.size() / ((g.event_ts.max() - g.event_ts.min()).dt.total_seconds() + 1)

    hours = pd.crosstab(ev.cookie_id, ev.event_ts.dt.hour)
    out['n_active_hours'] = (hours > 0).sum(axis=1)
    out['night_share'] = (hours.reindex(columns=range(24), fill_value=0)
                          .iloc[:, 0:7].sum(axis=1) / hours.sum(axis=1))
    return out


def f_div(ev):
    """Насколько разнообразно то, что кука смотрит."""
    g = ev.groupby('cookie_id')
    n = g.size()

    out = pd.DataFrame({
        'u_cat': g.item_category.nunique(),
        'u_loc': g.item_location.nunique(),
        'u_query': g.search_query.nunique(),
        'sp_max': g.search_page.max(),
        'sp_mean': g.search_page.mean(),
    })
    out['u_item_ratio'] = g.item_id.nunique() / n
    out['u_cat_ratio'] = out.u_cat / n
    out['u_loc_ratio'] = out.u_loc / n
    out['cat_entropy'] = pd.crosstab(ev.cookie_id, ev.item_category).apply(entropy, axis=1)
    return out

In [ ]:
def longest_run(flags):
    """Длина самой длинной серии подряд идущих True."""
    best = current = 0
    for f in flags:
        current = current + 1 if f else 0
        best = max(best, current)
    return best


def f_rhythm(ev):
    """Структура интервалов: повторяемость шага, серии, сессии.

    Человек делает паузы разной длины, скрипт часто ходит с почти одинаковым шагом.
    """
    dt = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()
    x = ev[['cookie_id']].assign(dt=dt)
    g = x.dropna(subset=['dt']).groupby('cookie_id')
    n = g.size()

    out = pd.DataFrame(index=n.index)
    out['dt_mode_share'] = g.dt.apply(lambda s: s.value_counts().iloc[0] / len(s))
    out['dt_uniq_ratio'] = g.dt.nunique() / n
    out['dt_entropy'] = g.dt.apply(lambda s: entropy(s.value_counts()))
    out['dt_metronome'] = g.dt.apply(lambda s: (np.abs(s - s.median()) <= 0.1 * max(s.median(), 1)).mean())
    out['dt_iqr'] = g.dt.quantile(0.75) - g.dt.quantile(0.25)
    out['dt_iqr_ratio'] = out.dt_iqr / g.dt.median().replace(0, np.nan)
    out['log_dt_std'] = g.dt.apply(lambda s: np.log1p(s).std())

    # сессия обрывается паузой больше 10 минут
    marks = x.assign(is_new=(x.dt.isna()) | (x.dt > 600))
    session_id = marks.groupby('cookie_id').is_new.cumsum()
    sizes = marks.assign(sid=session_id).groupby(['cookie_id', 'sid']).size()
    out['n_sessions'] = sizes.groupby('cookie_id').size()
    out['ses_max'] = sizes.groupby('cookie_id').max()
    out['ses_mean'] = sizes.groupby('cookie_id').mean()

    out['fast_maxrun'] = x.assign(f=(x.dt < 5).fillna(False)).groupby('cookie_id').f.apply(longest_run)
    out['sec_uniq_ratio'] = ev.groupby('cookie_id').event_ts.apply(lambda s: s.dt.second.nunique() / len(s))
    return out


def f_ptr(ev):
    """Поведение курсора. Координаты приходят только с web, поэтому считаем по web-событиям."""
    web = ev[ev.platform == 'web']
    pointed = web.dropna(subset=['pointer_x'])
    g = pointed.groupby('cookie_id')

    out = pd.DataFrame({
        'ptr_x_std': g.pointer_x.std(),
        'ptr_y_std': g.pointer_y.std(),
        'ptr_x_mean': g.pointer_x.mean(),
        'ptr_y_mean': g.pointer_y.mean(),
    })
    positions = pointed.assign(pos=pointed.pointer_x.astype(str) + '_' + pointed.pointer_y.astype(str))
    out['ptr_uniq_ratio'] = positions.groupby('cookie_id').pos.nunique() / g.size()
    # доля web-событий куки, у которых координаты вообще есть
    out['ptr_cover'] = g.size() / web.groupby('cookie_id').size()
    return out


def f_ua(ev):
    """Сколько разных платформ под одной кукой."""
    return pd.DataFrame({'n_platform': ev.groupby('cookie_id').platform.nunique()})


FEATURE_BLOCKS = {'base': f_base, 'time': f_time, 'div': f_div,
                  'ua': f_ua, 'rhythm': f_rhythm, 'ptr': f_ptr}

In [ ]:
def build_features(ev, meta, vocab=None):
    parts = {name: fn(ev) for name, fn in FEATURE_BLOCKS.items()}
    block_cols = {name: list(p.columns) for name, p in parts.items()}

    # мерджим от метаданных, иначе число строк зависит от наличия событий
    X = pd.concat(parts.values(), axis=1).reindex(meta.cookie_id)

    age = (meta.window_start_ts - meta.cookie_created_at).dt.total_seconds() / 86400
    X['cookie_age_d'] = age.values
    block_cols['base'].append('cookie_age_d')

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if vocab is None:
        vocab = list(X.columns)
    return X.reindex(columns=vocab, fill_value=0.0), vocab, block_cols


_, _, BLOCK_COLS = build_features(ev_train, train)
{k: len(v) for k, v in BLOCK_COLS.items()}

## Валидация

In [ ]:
y = train.target.values

# одиночный holdout: тот же порог, что в quickstart, для сравнимости baseline
VALID_FROM = '2026-04-17'
is_valid = train.window_start_ts.ge(VALID_FROM).values

print(f'обучение : {(~is_valid).sum()} кук, ботов {y[~is_valid].sum()}')
print(f'валидация: {is_valid.sum()} кук, ботов {y[is_valid].sum()}')

In [ ]:
# скользящие фолды: обучение на всём прошлом, валидация на следующих двух днях
FOLD_EDGES = [('2026-04-12', '2026-04-14'), ('2026-04-14', '2026-04-16'),
              ('2026-04-16', '2026-04-18'), ('2026-04-18', '2026-04-20')]

# признаки считаются один раз на фолд, дальше конфигурации только выбирают колонки
FOLD_DATA = []
for start, end in FOLD_EDGES:
    fit_mask = train.window_start_ts.lt(start).values
    val_mask = (train.window_start_ts.ge(start) & train.window_start_ts.lt(end)).values

    ev_fit = ev_train[ev_train.cookie_id.isin(train.cookie_id[fit_mask])]
    ev_val = ev_train[ev_train.cookie_id.isin(train.cookie_id[val_mask])]
    X_fit, vocab, _ = build_features(ev_fit, train[fit_mask])
    X_val, _, _ = build_features(ev_val, train[val_mask], vocab)

    FOLD_DATA.append((X_fit, y[fit_mask], X_val, y[val_mask]))
    print(f'обучение {fit_mask.sum():5d} (ботов {y[fit_mask].sum():3d}) | '
          f'валидация {val_mask.sum():5d} (ботов {y[val_mask].sum():3d})')

In [ ]:
def make_lr():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=2000, class_weight='balanced',
                                  random_state=RANDOM_STATE)),
    ])


def make_gb():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                          max_leaf_nodes=31, l2_regularization=1.0,
                                          random_state=RANDOM_STATE)


def cv_score(blocks, make_model):
    wanted = [c for b in blocks for c in BLOCK_COLS[b]]
    scores = []
    for X_fit, y_fit, X_val, y_val in FOLD_DATA:
        cols = [c for c in wanted if c in X_fit.columns]
        model = make_model().fit(X_fit[cols], y_fit)
        scores.append(precision_at_recall(y_val, model.predict_proba(X_val[cols])[:, 1]))
    return np.array(scores)

## Baseline

Логрег на объёме и долях активности, одиночный holdout.

In [ ]:
ev_fit = ev_train[ev_train.cookie_id.isin(train.cookie_id[~is_valid])]
ev_val = ev_train[ev_train.cookie_id.isin(train.cookie_id[is_valid])]

X_fit, VOCAB, _ = build_features(ev_fit, train[~is_valid])
X_val, _, _ = build_features(ev_val, train[is_valid], VOCAB)

base_cols = BLOCK_COLS['base']
baseline_model = make_lr().fit(X_fit[base_cols], y[~is_valid])
BASELINE_SCORE = precision_at_recall(y[is_valid], baseline_model.predict_proba(X_val[base_cols])[:, 1])

print('baseline P@R0.7:', round(BASELINE_SCORE, 4))

In [ ]:
def quickstart_features(ev, meta):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({'n_events': g.size(), 'item_nunique': g.item_id.nunique()})
    return meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left').fillna(0)


qs_cols = ['n_events', 'item_nunique']
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE)
rf.fit(quickstart_features(ev_fit, train[~is_valid])[qs_cols], y[~is_valid])
qs_score = precision_at_recall(
    y[is_valid], rf.predict_proba(quickstart_features(ev_val, train[is_valid])[qs_cols])[:, 1])

pd.DataFrame({
    'подход': ['константа', 'quickstart RF', 'baseline logreg'],
    'P@R0.7': [y[is_valid].mean(), qs_score, BASELINE_SCORE],
}).round(4)

Baseline отправлялся на лидерборд: публичный скор 0.176, константа на тесте 0.094.
Локальная оценка совпала с публичной в пределах процентного пункта, валидация не смещена.

## Блоки признаков на фолдах

In [ ]:
COMBOS = [
    ['base'],
    ['base', 'time'],
    ['base', 'div'],
    ['base', 'time', 'div'],
    ['base', 'time', 'div', 'ua'],
    ['base', 'time', 'div', 'ua', 'ptr'],
    ['base', 'time', 'div', 'ua', 'ptr', 'rhythm'],
]

rows = []
for blocks in COMBOS:
    row = {'признаки': '+'.join(blocks)}
    for name, maker in [('logreg', make_lr), ('boosting', make_gb)]:
        s = cv_score(blocks, maker)
        row[name] = round(s.mean(), 4)
        row[name + '_std'] = round(s.std(), 4)
    rows.append(row)

pd.DataFrame(rows)

Временные признаки для логрега бесполезны, для бустинга дают почти удвоение: связь
немонотонная, подозрительна середина распределения интервалов, а не край.

Признаки курсора дают самый большой прирост и заодно втрое сокращают разброс по фолдам.

Отдельно проверялись блоки по листанию выдачи и по разбору `user_agent` на семейства
браузера и ОС: прирост в пределах погрешности, в финальный набор не вошли. Также
удалены признаки с отрицательной перестановочной важностью (`span_sec`, `u_item`,
`item_repeat`, `hour_entropy`, `dt_std`, `frac_dt_lt5`, `u_query_ratio`, `n_ua`).

## Финальная модель

In [ ]:
FINAL_BLOCKS = ['base', 'time', 'div', 'ua', 'ptr', 'rhythm']
FINAL_COLS = [c for b in FINAL_BLOCKS for c in BLOCK_COLS[b]]

final_cv = cv_score(FINAL_BLOCKS, make_gb)
print(f'финальная модель: {final_cv.mean():.4f} +- {final_cv.std():.4f}')
print('по фолдам:', np.round(final_cv, 4))
print(f'baseline (holdout): {BASELINE_SCORE:.4f}')

In [ ]:
X_f, y_f, X_v, y_v = FOLD_DATA[-1]
cols = [c for c in FINAL_COLS if c in X_f.columns]
probe = make_gb().fit(X_f[cols], y_f)

imp = permutation_importance(
    probe, X_v[cols], y_v, n_repeats=5, random_state=RANDOM_STATE,
    scoring=lambda est, X, t: precision_at_recall(t, est.predict_proba(X)[:, 1]),
)
pd.Series(imp.importances_mean, index=cols).sort_values(ascending=False).head(15).round(4)

## Курсор: проверка на подмену платформы

Координаты есть только на web, поэтому нули в `ptr_*` могли бы просто кодировать «мобайл».
Смотрим только на куки, у которых почти вся активность на web.

In [ ]:
X_all, _, _ = build_features(ev_train, train)
web_events = ev_train[ev_train.platform == 'web'].groupby('cookie_id').size()
web_share = (web_events / ev_train.groupby('cookie_id').size()).reindex(train.cookie_id).fillna(0)

check = X_all.assign(target=y, web_share=web_share.values)
mostly_web = check.web_share > 0.9

print('кук:', mostly_web.sum(), '| из них ботов:', int(check[mostly_web].target.sum()))
check[mostly_web].groupby('target')[['ptr_x_std', 'ptr_y_std', 'ptr_uniq_ratio', 'ptr_cover']].median().round(3)

In [ ]:
# сколько активности ботов вообще приходится на web
check.groupby('target').web_share.mean().round(3)

Внутри web медианы всех четырёх признаков курсора у ботов равны нулю: веб-боты не
порождают координат вообще. Это не прокси платформы, платформа остаётся отдельным
признаком со своей важностью.

Отсюда и потолок метрики: примерно треть активности ботов приходится на android и ios,
где координат нет физически. Веб-ботов признак ловит почти даром, а оставшуюся полноту
до 70% приходится добирать на мобильных, где сигнал слабее.

## Сабмит

In [ ]:
X_full, VOCAB_FULL, _ = build_features(ev_train, train)
X_test, _, _ = build_features(ev_test, test, VOCAB_FULL)
cols = [c for c in FINAL_COLS if c in X_full.columns]

final_model = make_gb().fit(X_full[cols], y)

sub = pd.DataFrame({
    'cookie_id': test.cookie_id.values,
    'score': final_model.predict_proba(X_test[cols])[:, 1],
})

assert len(sub) == len(test)
assert set(sub.cookie_id) == set(test.cookie_id)
assert sub.cookie_id.duplicated().sum() == 0
assert sub.score.notna().all() and sub.score.between(0, 1).all()

sub.to_csv('submission.csv', index=False)
print(sub.shape, round(sub.score.min(), 4), round(sub.score.max(), 4))
sub.head()